# Student Depression Prediction Using Machine Learning

SENG 352 Data Analysis term project. This notebook builds a reproducible binary classification workflow for the Kaggle Student Depression Dataset.

Target variable: `Depression` (`0` = No Depression, `1` = Depression).

All random processes use `random_state = 42`.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.append(str(PROJECT_ROOT))

from src.project_pipeline import (
    RANDOM_STATE, TARGET, DATA_PATH, ensure_directories, load_dataset, validate_target,
    data_quality_report, clean_dataset, save_eda_plots, split_features,
    compare_feature_engineering, compare_models, tune_models, evaluate_final_model, save_model,
)

sns.set_theme(style="whitegrid")
ensure_directories()
print(f"Project root: {PROJECT_ROOT}")
print(f"Random state: {RANDOM_STATE}")

## 1. Data Loading

The dataset is loaded from `data/raw/`. The first checks verify the expected size, column names, data types, first rows, summary statistics, and the binary target column.

In [ ]:
df = load_dataset(DATA_PATH)
print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())
print("\nData types:")
display(df.dtypes)
print("\nFirst rows:")
display(df.head())
print("\nSummary statistics:")
display(df.describe(include="all").T)

validation = validate_target(df)
validation

In [ ]:
assert 27000 <= df.shape[0] <= 29000, "Unexpected row count; verify the input CSV."
assert df.shape[1] == 18, "Unexpected feature count; verify the input CSV."
assert validation["target_exists"], "Target column Depression is missing."
assert validation["is_binary_target"], "Depression must contain only binary values 0 and 1."

## 2. Data Quality Assessment

This section checks missing values, duplicate rows, rare categories, unusual numeric values, and IQR outlier counts. Outliers are inspected rather than blindly removed because unusual values can still be real responses or survey coding artifacts.

In [ ]:
dqa = data_quality_report(df)

missing = pd.DataFrame({
    "missing_count": pd.Series(dqa["missing"]),
    "missing_pct": pd.Series(dqa["missing"]) / len(df) * 100,
}).sort_values("missing_count", ascending=False)
display(missing)
print("Financial Stress missing values:", dqa["financial_stress_missing"])
print("Duplicate rows:", dqa["duplicate_rows"])
print("Rows with CGPA = 0:", dqa["cgpa_zero_rows"])
print("Rows with Age >= 59:", dqa["age_59_or_higher_rows"])

In [ ]:
outliers = pd.DataFrame(dqa["numeric_outlier_counts_iqr"].items(), columns=["feature", "iqr_outlier_count"])
display(outliers.sort_values("iqr_outlier_count", ascending=False))

print("Rare category checks:")
for col, counts in dqa["rare_categories"].items():
    print(f"\n{col}")
    display(pd.Series(counts, name="count"))

print("Near-zero-variance checks:")
display(pd.DataFrame(dqa["near_zero_variance"]).T)

### Data Quality Decisions

Rows with missing `Financial Stress` are dropped only if the count is very small. Rare `Others` categories in `Sleep Duration` and `Dietary Habits` are kept because they represent valid survey responses and one-hot encoding can handle them. `CGPA = 0` and high ages are inspected but not automatically removed. `Work Pressure` and `Job Satisfaction` are dropped during modeling after the near-zero-variance evidence confirms they are non-informative in this student-only dataset.

In [ ]:
clean_df = clean_dataset(df)
print("Clean shape:", clean_df.shape)
clean_df.to_csv(PROJECT_ROOT / "data/processed/student_depression_clean.csv", index=False)
display(clean_df.head())

## 3. Exploratory Data Analysis

The plots below are saved into `figures/`. The analysis focuses especially on academic pressure, financial stress, sleep duration, work/study hours, CGPA, suicidal thoughts, and family history of mental illness.

In [ ]:
figure_paths = save_eda_plots(clean_df)
print(f"Saved {len(figure_paths)} figures")
for path in figure_paths:
    print(path)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=clean_df, x=TARGET, hue=TARGET, palette="Set2", legend=False)
plt.title("Target Distribution")
plt.show()

focus_cols = [
    "Academic Pressure", "Financial Stress", "Work/Study Hours", "CGPA",
    "Sleep Duration", "Have you ever had suicidal thoughts ?",
    "Family History of Mental Illness", TARGET
]
display(clean_df[[c for c in focus_cols if c in clean_df.columns]].head())

### EDA Interpretation

The target distribution shows whether class imbalance exists and motivates the use of stratified splitting and class-weighted models. Academic and financial stress variables are expected to be strong risk indicators. Sleep and lifestyle variables can interact with academic pressure, so they are considered in feature engineering rather than only as isolated predictors.

In [ ]:
numeric_focus = [c for c in ["Academic Pressure", "Financial Stress", "Work/Study Hours", "CGPA", "Age"] if c in clean_df.columns]
fig, axes = plt.subplots(1, len(numeric_focus), figsize=(4 * len(numeric_focus), 4))
if len(numeric_focus) == 1:
    axes = [axes]
for ax, col in zip(axes, numeric_focus):
    sns.boxplot(data=clean_df, x=TARGET, y=col, hue=TARGET, palette="Set2", legend=False, ax=ax)
    ax.set_title(f"{col} by Depression")
plt.tight_layout()
plt.show()

## 4. Preprocessing and Train-Test Split

The pipeline uses an 80/20 stratified train-test split. Numeric variables are scaled using `StandardScaler`, and nominal categorical variables are one-hot encoded. Scaling and encoding are fitted only on the training folds through the sklearn pipeline.

In [ ]:
X_train, X_test, y_train, y_test = split_features(clean_df)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train target distribution:")
display(y_train.value_counts(normalize=True).rename("proportion"))
print("Test target distribution:")
display(y_test.value_counts(normalize=True).rename("proportion"))

### Sleep Duration Encoding Decision

`Sleep Duration` is treated as nominal for the main preprocessing because `Others` has no reliable position in an ordinal scale. For the engineered interaction only, the ordered sleep ranges are converted into approximate hour estimates; `Others` is filled using the training median inside the transformer.

## 5. Feature Engineering

The workflow compares performance with and without engineered features. The required engineered feature is `Academic_Pressure_x_Sleep`, based on academic pressure multiplied by estimated sleep hours. Additional tested features include stress load, academic lifestyle load, sleep estimate, and CGPA category.

In [ ]:
feature_engineering_results = compare_feature_engineering(X_train, y_train)
feature_engineering_results.to_csv(PROJECT_ROOT / "feature_engineering_comparison.csv", index=False)
display(feature_engineering_results)

## 6. Modeling

Models are compared using the same train-test split and stratified 5-fold cross-validation. Metrics include accuracy, precision, recall, F1-score, macro F1-score, and ROC-AUC. Class imbalance is handled with `class_weight="balanced"` where applicable. XGBoost is included automatically when installed.

In [ ]:
model_comparison = compare_models(X_train, y_train)
display(model_comparison)

## 7. Hyperparameter Tuning

The best two promising tunable models are selected from the cross-validation table and tuned with `GridSearchCV` using stratified 5-fold cross-validation. The primary scoring metric is macro F1-score.

In [ ]:
top_model_names = model_comparison["model"].tolist()
tuning_results, best_model = tune_models(X_train, y_train, top_model_names)
display(tuning_results)

## 8. Final Evaluation

The selected tuned model is evaluated once on the held-out test set. This section saves the classification report, confusion matrix, ROC curve, precision-recall curve, feature importance or permutation importance, and final metric CSV files.

In [ ]:
final_metrics, evaluation_details = evaluate_final_model(best_model, X_test, y_test)
display(final_metrics)
print("Confusion matrix:")
print(np.array(evaluation_details["confusion_matrix"]))

model_path = save_model(best_model)
print("Saved final model to:", model_path)

### Model Selection Discussion

The final model is selected based primarily on cross-validated macro F1-score, with ROC-AUC used as a secondary measure. In depression prediction, recall is especially important because a false negative means a student may be at risk but the model predicts no depression. Increasing recall can produce more false positives, but in an early-warning system that trade-off may be acceptable if follow-up support is non-punitive and reviewed by qualified humans.

## 9. Error Analysis

False positives and false negatives are saved to `reports/misclassified_samples.csv`, and group-level error summaries are saved in `reports/`. False negatives deserve special attention because they correspond to potentially at-risk students being missed.

In [ ]:
print("False positives:", evaluation_details["errors"]["false_positives"])
print("False negatives:", evaluation_details["errors"]["false_negatives"])

misclassified = pd.read_csv(PROJECT_ROOT / "reports/misclassified_samples.csv")
display(misclassified.head())

for name, table in evaluation_details["errors"]["group_error_tables"].items():
    print(f"\nError rates by {name}")
    display(pd.DataFrame(table))

### Ethical Limitation

This model must not be used as a clinical diagnostic tool. It can only be considered as a decision-support or early-warning prototype. Any real-world use would require expert review, privacy protection, bias monitoring, careful consent and governance, and institutional approval.

## 10. Final Checklist

The notebook loads the dataset, validates the target, handles missing values, saves EDA plots, builds leakage-safe preprocessing pipelines, compares at least four models, performs cross-validation, tunes the best models, evaluates a final model on the held-out test set, saves the final model, saves metric CSV files, discusses false negatives, and includes ethical limitations.